In [ ]:
import os
import re
import random
import pandas as pd
from transformers import pipeline
from deep_translator import GoogleTranslator
from transformers.utils import logging

logging.set_verbosity_error()
os.environ["TRANSFORMERS_NO_AE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TELEMETRY"] = "YES"

TARGET = 40000
BATCH_SIZE = 8
BACK_TRANSLATE_PROB = 0.15

df = pd.read_csv(r"E:\nachiket\RP\Dataset-SA.csv")

pos_df = df[df["Sentiment"] == "positive"]
neg_df = df[df["Sentiment"] == "negative"]
neu_df = df[df["Sentiment"] == "neutral"]

paraphraser = pipeline(
    "text2text-generation",
    model="t5-small",
    tokenizer="t5-small",
    device=-1   # CPU ONLY
)

_ = paraphraser("paraphrase: warmup sentence", max_new_tokens=8)

def is_valid_paraphrase(original, new):
    if not new: return False
    if new.strip().lower() == original.strip().lower(): return False
    o = re.sub(r"[^a-zA-Z0-9 ]","", original.lower())
    n = re.sub(r"[^a-zA-Z0-9 ]","", new.lower())
    if n in o or o in n: return False
    if len(set(n.split())) < 3: return False
    return True

def generate_diverse_paraphrase(text):
    try:
        out = paraphraser(
            "paraphrase: " + text,
            max_new_tokens=40,
            do_sample=True,
            top_k=40,
            top_p=0.90,
            temperature=0.90,
            repetition_penalty=1.2,
            num_return_sequences=1,
            truncation=True
        )
        return out[0]["generated_text"].strip()
    except:
        return ""

def paraphrase_batch(texts):
    final = []
    for orig in texts:
        new = generate_diverse_paraphrase(orig)
        if not is_valid_paraphrase(orig, new):
            new = generate_diverse_paraphrase(orig)
        if not is_valid_paraphrase(orig, new):
            words = orig.split()
            if len(words) > 3:
                i = random.randint(0, len(words)-2)
                words[i], words[i+1] = words[i+1], words[i]
                new = " ".join(words)
            else:
                new = orig + " overall it works fine."
        final.append(new)
    return final

def back_translate(text):
    try:
        hi = GoogleTranslator(source="en", target="hi").translate(text)
        en = GoogleTranslator(source="hi", target="en").translate(hi)
        if not en or en.lower() == text.lower():
            en = "In my opinion, " + text
        return en.strip().capitalize()
    except:
        return "In my view, " + text

hinglish_map = {
    "good":"acha","great":"mast","bad":"bekar",
    "excellent":"jabardast","awesome":"bhot badhiya",
    "poor":"kharab","product":"item","nice":"sahi",
    "very":"bahut","super":"superb","worst":"sabse bekar"
}

def to_hinglish(text):
    return " ".join(hinglish_map.get(w.lower(), w) for w in text.split())

def augment_class(df_class, label):
    current = len(df_class)
    needed = TARGET - current

    if needed <= 0:
        print(f"{label}: already {current}, skipping.")
        return pd.DataFrame()

    print(f"{label}: need {needed}")

    augmented = []
    temp_batch = []
    count = 0

    while len(augmented) < needed:
        s = df_class.sample(1).iloc[0]
        text = random.choice([str(s.get("Review","")), str(s.get("Summary",""))])

        if random.random() < BACK_TRANSLATE_PROB:
            aug_type = "back"
        else:
            aug_type = random.choice(["para", "hing"])

        if aug_type == "para":
            temp_batch.append((s,text))
            if len(temp_batch) >= BATCH_SIZE:
                new_texts = paraphrase_batch([t[1] for t in temp_batch])
                for (sample,_), new in zip(temp_batch, new_texts):
                    row = sample.copy()
                    row["Review"] = new
                    augmented.append(row)
                    count += 1
                    if count % 100 == 0:
                        print(f"{label}: {count}/{needed}")
                temp_batch = []

        elif aug_type == "back":
            new = back_translate(text)
            row = s.copy()
            row["Review"] = new
            augmented.append(row)
            count += 1
            if count % 100 == 0:
                print(f"{label}: {count}/{needed}")

        else:
            new = to_hinglish(text)
            row = s.copy()
            row["Review"] = new
            augmented.append(row)
            count += 1
            if count % 100 == 0:
                print(f"{label}: {count}/{needed}")

    return pd.DataFrame(augmented[:needed])

aug_pos = augment_class(pos_df, "positive")
aug_neg = augment_class(neg_df, "negative")
aug_neu = augment_class(neu_df, "neutral")

final = pd.concat([df, aug_pos, aug_neg, aug_neu], ignore_index=True)
final = final.sample(frac=1, random_state=42).reset_index(drop=True)

final.to_csv(r"E:\nachiket\RP\Dataset-SA-Augmented-40000.csv", index=False, encoding="utf-8-sig")

print("Done:", final.shape)
print("Saved to: C:\\rp\\Dataset-SA-Augmented-40000.csv")
